In [1]:
import pandas as pd
import os, dill
import numpy as np
import cvxpy as cp
from typing import Iterable, Optional, Mapping, cast
from plotly import graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px

from ecoli.processes.metabolism_redux_classic import NetworkFlowModel, FlowResult

os.chdir(os.path.expanduser('~/dev/vEcoli'))

%load_ext autoreload

In [2]:
# import sim
time = '600'
date = '2026-01-22'
experiment = 'homeostatic_only'
condition = 'basal'
entry = f'{experiment}_{time}_{date}'
folder = f'out/objective_weight/{condition}/{entry}/'

output = np.load(folder + '0_output.npy',allow_pickle='TRUE').item()
output = output['agents']['0']
fba = output['listeners']['fba_results']
bulk = pd.DataFrame(output['bulk'])
f = open(folder + 'agent_steps.pkl', 'rb')
agent = dill.load(f)
f.close()

metabolism = agent['ecoli-metabolism-redux-classic']

In [3]:
stoichiometry = metabolism.stoichiometry.copy()
reaction_names = metabolism.reaction_names
metabolites = metabolism.metabolite_names.copy()
counts_to_molar = output['listeners']['enzyme_kinetics']['counts_to_molar']

homeostatic_only_term = np.array(fba['homeostatic_term'].copy())/counts_to_molar # covert to counts
homeostatic_only_baseline = np.max(homeostatic_only_term) # in counts

homeostatic_dm_targets =  pd.DataFrame(fba['target_homeostatic_dmdt'], columns=metabolism.homeostatic_metabolites).mul(counts_to_molar, axis=0).max(axis=0)
homeostatic_metabolite_counts = pd.DataFrame(fba['homeostatic_metabolite_counts'], columns=metabolism.homeostatic_metabolites).mul(counts_to_molar, axis=0).mean(axis=0)
maintenance = pd.DataFrame(fba["maintenance_target"][1:], columns=['maintenance_reaction']).mul(counts_to_molar[1:], axis=0).mean(axis=0)
kinetic = pd.DataFrame(fba["target_kinetic_fluxes"], columns=metabolism.kinetic_constraint_reactions).mul(counts_to_molar, axis=0).mean(axis=0)

print(len(homeostatic_dm_targets))
len(homeostatic_metabolite_counts)

172


172

In [4]:
FREE_RXNS = [
    "TRANS-RXN-145",
    "TRANS-RXN0-545",
    "TRANS-RXN0-474",
    "ATPSYN-RXN (reverse)",
]

def test_NetworkFlowModel(weights, solver_choice = cp.GLOP,
                          homeostatic_metabolite_counts = homeostatic_metabolite_counts,
                          homeostatic_dm_targets = homeostatic_dm_targets,
                          kinetic=kinetic,
                          binary_kinetics_idx=None,
                          maintenance=maintenance,
                          ):
    model = NetworkFlowModel(
            stoich_arr=stoichiometry,
            metabolites=metabolites,
            reactions=reaction_names,
            homeostatic_metabolites=metabolism.homeostatic_metabolites,
            kinetic_reactions=metabolism.kinetic_constraint_reactions,
            free_reactions=FREE_RXNS)
    model.set_up_exchanges(exchanges=metabolism.exchange_molecules, uptakes=metabolism.allowed_exchange_uptake)
    solution: FlowResult = model.solve(
            homeostatic_concs=homeostatic_metabolite_counts, # in conc
            homeostatic_dm_targets=np.array(list(dict(homeostatic_dm_targets).values())), # conc
            maintenance_target=maintenance, # conc range
            kinetic_targets=np.array(list(dict(kinetic).values())), # conc range
            binary_kinetic_idx=binary_kinetics_idx,
            # force_flow_idx=force_reaction_idx,
            objective_weights=weights, #same
            upper_flux_bound= 100, # increase to 10^9 because notebook runs FlowResult using Counts if directly imported from listeners, WC runs using conc.
            solver=solver_choice) #SCS. ECOS, MOSEK
    return solution.objective, solution.homeostatic_term, solution.secretion_term, solution.velocities, solution.dm_dt

In [5]:
def met_homeo(MOI, dmdt):
    ddt = dmdt.loc[MOI]
    return np.abs(ddt-homeostatic_dm_targets.loc[MOI])/homeostatic_metabolite_counts[MOI]

# Pairwise with Efficiency

In [ ]:
lambda_eff_range = np.logspace(-10, -4, 1000)
acceptable_lambda = []
homeo_diff = []
tol = 0.1
pareto = dict()

for lam in lambda_eff_range:
    objective_weights = {
        "efficiency": lam,
        "homeostatic": 1,
    }

    try :
        obj_val, home_term, eff_term, v, dmdt = test_NetworkFlowModel(objective_weights)

        pareto[lam] = dict()
        pareto[lam]['Homeostatic Term'] = home_term
        pareto[lam]['Weighted Efficiency Term'] = eff_term
        pareto[lam]['Unweighted Efficiency Term'] = eff_term/lam

        flux = pd.DataFrame(v, index=reaction_names)
        met_dt = pd.DataFrame(dmdt, index=metabolism.metabolite_names)

        CPD12261_dt = met_dt.loc['CPD-12261[p]']
        glycomono_dt = met_dt.loc["glycogen-monomer[c]"]

        CPD12261_home = met_homeo('CPD-12261[p]', met_dt)
        permitted_home = home_term - met_homeo("glycogen-monomer[c]", met_dt)

        if np.isclose(CPD12261_home, permitted_home):
            acceptable_lambda.append(lam)
    except:
        continue

## Pareto Curve with Efficiency

In [ ]:
pareto_df = (
    pd.DataFrame.from_dict(pareto, orient="index")
      .rename_axis("lambda_eff")
      .reset_index()
      .sort_values("lambda_eff")
)

# Safety: drop any weird values
pareto_df = pareto_df.replace([np.inf, -np.inf], np.nan).dropna()


In [ ]:
import plotly.express as px
import plotly.express.colors as pc

# Pastel palette (qualitative). Take the first two colors as endpoints.
cmin, cmax = pc.qualitative.Pastel[0], pc.qualitative.Pastel[2]

df = pareto_df.copy()
df = df.sort_values("lambda_eff")

# # Axes (typically log makes sense)
# fig.update_xaxes(type="log", title="Weighted Efficiency Term (eff_term / λ)")
#
# # Only log y if strictly > 0
# if (df["Homeostatic Term"] > 0).all():
#     fig.update_yaxes(type="log", title="Homeostatic Term")
# else:
#     fig.update_yaxes(title="Homeostatic Term")

# Make the colorbar log-scaled by transforming the color variable
# (recommended because λ spans orders of magnitude)
df["log10_lambda_eff"] = np.log10(df["lambda_eff"])
fig = px.scatter(
    df,
    x="Weighted Efficiency Term",
    y="Homeostatic Term",
    color="log10_lambda_eff",
    color_continuous_scale=[cmin, cmax],
    hover_data={"lambda_eff": ":.2e"},
    title="Pareto plot: Homeostatic vs Efficiency (colored by log10 λeff)",
)
fig.update_xaxes(type="log")
if (df["Homeostatic Term"] > 0).all():
    fig.update_yaxes(type="log")
fig.update_coloraxes(colorbar_title="log10(λeff)")

# Add pareto curve
best_y = np.inf
best_x = np.inf
frontier_mask = []
for i in range(len(df)):
    y = df.loc[i, "Homeostatic Term"]
    x = df.loc[i, "Weighted Efficiency Term"]
    if y < best_y:
        frontier_mask.append(True)
        best_y = y
    elif x < best_x:
        frontier_mask.append(True)
        best_x = x
    else:
        frontier_mask.append(False)

frontier = df[frontier_mask]

fig.add_trace(go.Scatter(
    x=frontier["Weighted Efficiency Term"],
    y=frontier["Homeostatic Term"],
    mode="lines+markers",
    line=dict(color=pc.qualitative.Pastel[1], dash="dot"),
    showlegend=False
))

# change order of trace
fig.data = (fig.data[1], fig.data[0])

fig.update_layout(
    template="plotly_white",
    # paper_bgcolor='rgba(255, 0, 0, 0)',
    # plot_bgcolor='rgba(255, 0, 0, 0)',
)
fig.show()
# fig.write_image(f'/Users/HeenaSaqib/dev/vEcoli/notebooks/Heena notebooks/Metabolism_New Genes/out/objective_weights/homeostatic_efficiency/'
#                 f'pareto_frontier.png', width=800, height=600, scale=3)

# Pairwise with Kinetics

In [6]:
binary_kinetic_idx = metabolism.binary_kinetic_idx

In [118]:
homeostatic_dm_targets =  pd.DataFrame(fba['target_homeostatic_dmdt'], columns=metabolism.homeostatic_metabolites).mul(counts_to_molar, axis=0).max(axis=0)
homeostatic_metabolite_counts = pd.DataFrame(fba['homeostatic_metabolite_counts'], columns=metabolism.homeostatic_metabolites).mul(counts_to_molar, axis=0).mean(axis=0)
maintenance = pd.DataFrame(fba["maintenance_target"][1:], columns=['maintenance_reaction']).mul(counts_to_molar[1:], axis=0).mean(axis=0)
kinetic = pd.DataFrame(fba["target_kinetic_fluxes"], columns=metabolism.kinetic_constraint_reactions).mul(counts_to_molar, axis=0).mean(axis=0)

n = 1000
lambda_range = np.logspace(-10, 0, n)
acceptable_lambda = []
homeo_diff = []
tol = 1E-7
pareto = dict()
homeo_baseline = 0
count = 0

for lam in lambda_range:
    objective_weights = {
        "kinetics": lam,
        "homeostatic": 1,
    }

    try :
        obj_val, home_term, kin_term, v, dmdt = test_NetworkFlowModel(objective_weights,
                                                                      homeostatic_metabolite_counts = homeostatic_metabolite_counts,
                                                                      homeostatic_dm_targets = homeostatic_dm_targets,
                                                                      kinetic=kinetic,
                                                                      maintenance=maintenance,
                                                                      binary_kinetics_idx=None)
        if lam == lambda_range[0]:
            homeo_baseline = home_term

        pareto[lam] = dict()
        pareto[lam]['Homeostatic Term'] = home_term
        pareto[lam]['Weighted Kinetics Term'] = kin_term
        pareto[lam]['Unweighted Kinetics Term'] = kin_term/lam

        flux = pd.DataFrame(v, index=reaction_names)
        met_dt = pd.DataFrame(dmdt, index=metabolism.metabolite_names)

        CPD12261_dt = met_dt.loc['CPD-12261[p]']
        glycomono_dt = met_dt.loc["glycogen-monomer[c]"]

        CPD12261_home = met_homeo('CPD-12261[p]', met_dt).values
        glycomono_home = met_homeo("glycogen-monomer[c]", met_dt).values
        permitted_home = home_term - glycomono_home

        if ((np.isclose(CPD12261_home, permitted_home) or CPD12261_home < counts_to_molar[1]) and
                np.isclose(home_term, homeo_baseline)):
            count+=1
            acceptable_lambda.append(lam)
    except:
        continue

In [119]:
pareto_df = (
    pd.DataFrame.from_dict(pareto, orient="index")
      .rename_axis("lambda_kin")
      .reset_index()
      .sort_values("lambda_kin")
)

# Safety: drop any weird values
pareto_df = pareto_df.replace([np.inf, -np.inf], np.nan).dropna()


In [196]:
import plotly.express as px
import plotly.express.colors as pc

# Pastel palette (qualitative). Take the first two colors as endpoints.
cmin, cmax = pc.qualitative.Pastel[0], pc.qualitative.Pastel[2]

df = pareto_df.copy()
df = df.sort_values("lambda_kin")

# Make the colorbar log-scaled by transforming the color variable
# (recommended because λ spans orders of magnitude)
df["log10_lambda_kin"] = np.log10(df["lambda_kin"])
fig = px.scatter(
    df,
    x="Weighted Kinetics Term",
    y="Homeostatic Term",
    color="log10_lambda_kin",
    color_continuous_scale=[cmin, cmax],
    hover_data={"lambda_kin": ":.2e"},
    title=f"Pareto plot: Homeostatic vs Kinetics (colored by log10 λkin) Mean all, Max dmdt, Binary=F, n = {n}",
)

fig.update_xaxes(type="log")
if (df["Homeostatic Term"] > 0).all():
    fig.update_yaxes(type="log")
fig.update_coloraxes(colorbar_title="log10(λkin)")
fig.update_xaxes(type="log")

# Add pareto curve
elbow = [np.inf, np.inf]
elbow_idx = 0
for i in range(len(df)):
    y = df.loc[i, "Homeostatic Term"]
    x = df.loc[i, "Weighted Kinetics Term"]

    if x <= elbow[0] and y <= elbow[1]:
        elbow = [x, y]
        elbow_idx = i

best_x = elbow[0]
best_y = elbow[1]

screen_y = np.inf
# split the graph into lowest x and lowest y comparison
# first 0-i is comparing y , i+1:end is comparing x
frontier_mask = [True] * len(df)

for i in np.arange(elbow_idx):
    y = df.loc[i, "Homeostatic Term"]

    if best_y < y < screen_y:
        frontier_mask[i] = True
        screen_y = y
    else:
        frontier_mask[i] = False

screen_x = 0

for i in np.arange(elbow_idx, len(df)):
    x = df.loc[i, "Weighted Kinetics Term"]

    if best_x > x:
        frontier_mask[i] = True
        best_x = x
    else:
        frontier_mask[i] = False

frontier = df.loc[frontier_mask].sort_values("Weighted Kinetics Term")

fig.add_trace(go.Scatter(
    x=frontier["Weighted Kinetics Term"],
    y=frontier["Homeostatic Term"],
    mode="lines+markers",
    line=dict(color=pc.qualitative.Pastel[1], dash="dot"),
    showlegend=False
))

# change order of trace
fig.data = (fig.data[1], fig.data[0])

fig.update_layout(
    template="plotly_white",
    paper_bgcolor='rgba(255, 0, 0, 0)',
    plot_bgcolor='rgba(255, 0, 0, 0)',
)
fig.show(renderer="browser")
# fig.write_image(f'/Users/HeenaSaqib/dev/vEcoli/notebooks/Heena notebooks/Metabolism_New Genes/out/objective_weights/homeostatic_kinetics/'
#                 f'pareto_frontier_max_dmdt.png', width=1200, height=600, scale=3)
# fig.write_html(f'/Users/HeenaSaqib/dev/vEcoli/notebooks/Heena notebooks/Metabolism_New Genes/out/objective_weights/homeostatic_kinetics/'
#                 f'pareto_frontier_max_dmdt.html')

# Pairewise with Secretion

In [85]:
homeostatic_dm_targets =  pd.DataFrame(fba['target_homeostatic_dmdt'], columns=metabolism.homeostatic_metabolites).mul(counts_to_molar, axis=0).max(axis=0)
homeostatic_metabolite_counts = pd.DataFrame(fba['homeostatic_metabolite_counts'], columns=metabolism.homeostatic_metabolites).mul(counts_to_molar, axis=0).mean(axis=0)
maintenance = pd.DataFrame(fba["maintenance_target"][1:], columns=['maintenance_reaction']).mul(counts_to_molar[1:], axis=0).mean(axis=0)
kinetic = pd.DataFrame(fba["target_kinetic_fluxes"], columns=metabolism.kinetic_constraint_reactions).mul(counts_to_molar, axis=0).mean(axis=0)
binary_kinetic_idx = metabolism.binary_kinetic_idx

n = 50
lambda_secretion_range = np.logspace(-10, 0, n)
acceptable_lambda = []
homeo_diff = []
tol = 0.1
pareto = dict()

for lam in lambda_secretion_range:
    objective_weights = {
        "secretion": lam,
        "homeostatic": 1,
    }

    try :
        obj_val, home_term, secretion_term, v, dmdt = test_NetworkFlowModel(objective_weights,
                                                                            binary_kinetics_idx=None)

        pareto[lam] = dict()
        pareto[lam]['Homeostatic Term'] = home_term
        pareto[lam]['Weighted Secretion Term'] = secretion_term
        pareto[lam]['Unweighted Secretion Term'] = secretion_term/lam

        flux = pd.DataFrame(v, index=reaction_names)
        met_dt = pd.DataFrame(dmdt, index=metabolism.metabolite_names)

        CPD12261_dt = met_dt.loc['CPD-12261[p]']
        glycomono_dt = met_dt.loc["glycogen-monomer[c]"]

        CPD12261_home = met_homeo('CPD-12261[p]', met_dt)
        permitted_home = home_term - met_homeo("glycogen-monomer[c]", met_dt)

        if np.isclose(CPD12261_home, permitted_home):
            acceptable_lambda.append(lam)
    except:
        continue

In [86]:
pareto_df = (
    pd.DataFrame.from_dict(pareto, orient="index")
      .rename_axis("lambda_secretion")
      .reset_index()
      .sort_values("lambda_secretion")
)

# Safety: drop any weird values
pareto_df = pareto_df.replace([np.inf, -np.inf], np.nan).dropna()


In [94]:
import plotly.express as px
import plotly.express.colors as pc

# Pastel palette (qualitative). Take the first two colors as endpoints.
cmin, cmax = pc.qualitative.Pastel[0], pc.qualitative.Pastel[2]

df = pareto_df.copy()
df = df.sort_values("lambda_secretion")

# Make the colorbar log-scaled by transforming the color variable
# (recommended because λ spans orders of magnitude)
df["log10_lambda_secretion"] = np.log10(df["lambda_secretion"])
fig = px.scatter(
    df,
    x="Unweighted Secretion Term",
    y="Homeostatic Term",
    color="log10_lambda_secretion",
    color_continuous_scale=[cmin, cmax],
    hover_data={"lambda_secretion": ":.2e"},
    title=f"Pareto plot: Homeostatic vs Secretion (colored by log10 λsec) Mean all, Max dmdt, Binary=F, n = {n}",
)

fig.update_xaxes(type="log")
if (df["Homeostatic Term"] > 0).all():
    fig.update_yaxes(type="log")
fig.update_coloraxes(colorbar_title="log10(λsec)")
fig.update_xaxes(type="log")

## ------------ Calculate pareto curve ------------ ##
elbow = [np.inf, np.inf]
elbow_idx = 0
for i in range(len(df)):
    y = df.loc[i, "Homeostatic Term"]
    x = df.loc[i, "Unweighted Secretion Term"]

    if x <= elbow[0] and y <= elbow[1]:
        elbow = [x, y]
        elbow_idx = i

best_x = elbow[0]
best_y = elbow[1]

screen_y = np.inf
# split the graph into lowest x and lowest y comparison
# first 0-i is comparing y , i+1:end is comparing x
frontier_mask = [True] * len(df)

for i in np.arange(elbow_idx):
    y = df.loc[i, "Homeostatic Term"]

    if best_y < y < screen_y:
        frontier_mask[i] = True
        screen_y = y
    else:
        frontier_mask[i] = False

screen_x = 0

for i in np.arange(elbow_idx, len(df)):
    x = df.loc[i, "Unweighted Secretion Term"]

    if best_x > x:
        frontier_mask[i] = True
        best_x = x
    else:
        frontier_mask[i] = False

frontier = df.loc[frontier_mask].sort_values("Unweighted Secretion Term")

## ------------ Plot pareto curve ------------ ##
fig.add_trace(go.Scatter(
    x=frontier["Unweighted Secretion Term"],
    y=frontier["Homeostatic Term"],
    mode="lines+markers",
    line=dict(color=pc.qualitative.Pastel[1], dash="dot"),
    showlegend=False
))

# change order of trace
fig.data = (fig.data[1], fig.data[0])

fig.update_layout(
    template="plotly_white",
    paper_bgcolor='rgba(255, 0, 0, 0)',
    plot_bgcolor='rgba(255, 0, 0, 0)',
)
# fig.show(renderer="browser")
fig.write_image(f'/Users/HeenaSaqib/dev/vEcoli/notebooks/Heena notebooks/Metabolism_New Genes/out/objective_weights/homeostatic_secretion/'
                f'pareto_frontier_50.png', width=1200, height=600, scale=3)
fig.write_html(f'/Users/HeenaSaqib/dev/vEcoli/notebooks/Heena notebooks/Metabolism_New Genes/out/objective_weights/homeostatic_secretion/'
                f'pareto_frontier_50.html')

In [92]:
homeostatic_term = pareto_df['Homeostatic Term']
np.argmin(homeostatic_term)
homeostatic_term[16]
lambda_secretion_range[16]

np.float64(1.8420699693267165e-07)

In [27]:
objective_weights = {
    'secretion': 0,
    'homeostatic': 1,
}
obj_val_0, home_term_0, secretion_term_0, v_0, dmdt_0 = test_NetworkFlowModel(objective_weights)

In [29]:
secretion_term_0, home_term_0

(np.float64(415.6207562462918), np.float64(3.7966743392377975e-10))

In [30]:
secretion_term_1, home_term_1

(np.float64(7.86408655005542), np.float64(7.73655408822365e-11))

In [75]:
# check variations is homeostatic dmdt across _0 and _1
home_dmdt_0 = pd.DataFrame(dmdt_0[metabolism.network_flow_model.homeostatic_idx], index = metabolism.homeostatic_metabolites) * 10000
home_dmdt_1 = pd.DataFrame(dmdt_1[metabolism.network_flow_model.homeostatic_idx], index = metabolism.homeostatic_metabolites) * 10000

not_close_boolean = ~ (home_dmdt_0 == home_dmdt_1)
home_dmdt_not_close_0 = home_dmdt_0[not_close_boolean]
home_dmdt_not_close_1 = home_dmdt_1[not_close_boolean]

In [72]:
home_dmdt_not_close_0

,0
2-3-DIHYDROXYBENZOATE[c],NaN
2-KETOGLUTARATE[c],1.583698
2-PG[c],0.407685
2K-4CH3-PENTANOATE[c],0.611527
4-AMINO-BUTYRATE[c],NaN
...,...
NA+[p],0.454725
OXYGEN-MOLECULE[p],0.536362
FE+3[p],0.747656
CA+2[p],0.568869


In [73]:
home_dmdt_0

,0
2-3-DIHYDROXYBENZOATE[c],0.611527
2-KETOGLUTARATE[c],1.583698
2-PG[c],0.407685
2K-4CH3-PENTANOATE[c],0.611527
4-AMINO-BUTYRATE[c],1.364175
...,...
NA+[p],0.454725
OXYGEN-MOLECULE[p],0.536362
FE+3[p],0.747656
CA+2[p],0.568869


In [17]:
[np.min(acceptable_lambda), np.max(acceptable_lambda)]

[np.float64(1e-10), np.float64(0.001648986944471065)]

In [10]:
[np.min(acceptable_lambda), np.max(acceptable_lambda)]

[np.float64(1e-10), np.float64(0.00121152765862859)]

In [11]:
acceptable_lambda

[np.float64(1e-10),
 np.float64(2.6101572156825333e-10),
 np.float64(6.812920690579622e-10),
 np.float64(1.7782794100389228e-09),
 np.float64(4.641588833612773e-09),
 np.float64(1.2115276586285901e-08),
 np.float64(3.162277660168379e-08),
 np.float64(8.25404185268019e-08),
 np.float64(2.1544346900318867e-07),
 np.float64(5.62341325190349e-07),
 np.float64(1.4677992676220705e-06),
 np.float64(3.8311868495572925e-06),
 np.float64(1e-05),
 np.float64(2.6101572156825386e-05),
 np.float64(6.812920690579622e-05),
 np.float64(0.00017782794100389227),
 np.float64(0.0004641588833612782),
 np.float64(0.00121152765862859)]